# YLT blend: annual aggregate loss

Weights $w$ for $F_w = \sum_m w_m F_m$ over STORM, CHAZ and IRIS annual-loss distributions, fitted to each target's
annual losses over `BASE_YEARS` by minimising the mean score (grid search). Data: `results/annual_losses_v2.csv`,
one row per simulated year; calendar year from the hazard file name.

| Section | What it runs |
|---|---|
| 1 | every target × standard scores |
| 2 | every target × quantile windows |
| 3 | 1 and 2 conditional on $X \le u$, with $u$ the largest IBTrACS annual loss (same $u$ for every target) |
| 4 | standard scores re-fitted on random subsets of whole years: unconditional and $X \le u$ |
| 5 | sections 1-3 repeated for each 2-model blend (STORM+CHAZ, STORM+IRIS, CHAZ+IRIS) |
| 6 | save all results |

**Rule sets.** `STANDARD`: CRPS, LogScore (discrete, bin width `LOG_BIN_WIDTH`), Quantile (0.05-0.95), a low/high
quantile triple, and QuantileWeightedCRPS. `WINDOWS`: QuantileScore on sliding windows of three quantile levels
(0.05-0.2, 0.1-0.3, ..., 0.8-0.95), showing how the weights move from low to high losses. Two windows coincide with
the STANDARD triples, so `{**STANDARD, **WINDOWS}` holds 13 distinct rules. Lower scores are better.

**Results** columns start `target, scoring_rule, weight_*`. Every result is stored in `results[<name>]`; the last
cell saves them all. Roughly 5-10 s per optimisation at `bc.GRID_STEP = 0.01` (`0.05` for a quick look).

In [7]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from dataclasses import replace
import scoring_rules_core as bc
from scoring_rules_core import *

data = load_annual_losses("results/annual_losses_v2.csv", "ENSO.csv")
RUNS_DIR, NOTES = "results/blend_runs", ""

COMPONENTS = ('STORM', 'CHAZ', 'IRIS')
TARGETS = ['IBTrACS', 'IBTrACS_p', 'MINI-WNC', 'MINI-WNC-QM']
BASE_YEARS = range(1980, 2021)
MIN_OBS = 5                      # specs with fewer target years are skipped (listed in df.attrs['skipped'])
LOG_BIN_WIDTH = 1e7              # LogScore bin width (loss units)

STANDARD = standard_rules(low_quantiles=(0.05, 0.1, 0.2), log_score_kwargs=dict(bin_width=LOG_BIN_WIDTH))
WINDOWS = quantile_window_rules()
BASE = BlendSpec(TARGETS[0], COMPONENTS, years=BASE_YEARS)
results = {}
summary(data)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,n,mean,median,max,frac_zero,year_min,year_max,n_years
model,,,,,,,,
CHAZ,9750,6.504654e+08,4.515763e+06,3.257631e+11,0.076615,NaN,NaN,0
IBTrACS,41,5.243460e+07,9.032616e+06,6.012146e+08,0.000000,1980.0,2020.0,41
IBTrACS_p,8000,9.100494e+07,7.127513e+06,7.900762e+09,0.001000,1980.0,2019.0,40
IRIS,10000,3.008080e+08,3.634150e+06,5.778755e+10,0.156700,NaN,NaN,0
MINI-WNC,8000,6.871185e+07,5.645596e+06,5.049862e+09,0.009625,1980.0,2019.0,40
MINI-WNC-QM,8000,6.729637e+07,4.373748e+06,5.571360e+09,0.009750,1980.0,2019.0,40
STORM,10000,1.165497e+08,2.785924e+06,3.281791e+10,0.082800,NaN,NaN,0


## 1. Standard scores

In [ ]:
results['standard'] = sweep(data, expand_specs(BASE, target=TARGETS), STANDARD, min_obs=MIN_OBS)
results['standard']


2/24 (8s)
4/24 (20s)
6/24 (32s)
8/24 (56s)
10/24 (101s)
12/24 (361s)
14/24 (387s)
16/24 (434s)
18/24 (720s)
20/24 (745s)
22/24 (794s)
24/24 (1089s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,years,n_years,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,CRPS,1.00,0.00,0.00,STORM,True,4.221732e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
1,IBTrACS,LogScore,1.00,0.00,0.00,STORM,True,2.271615e+00,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
2,IBTrACS,Quantile,1.00,0.00,0.00,STORM,True,2.178028e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
3,IBTrACS,"Quantile (0.05, 0.1, 0.2)",0.00,1.00,0.00,CHAZ,True,6.116608e+06,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",1.00,0.00,0.00,STORM,True,2.852634e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
5,IBTrACS,QuantileWeightedCRPS,1.00,0.00,0.00,STORM,True,1.817247e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
6,IBTrACS_p,CRPS,1.00,0.00,0.00,STORM,True,7.914704e+07,STORM+CHAZ+IRIS,1980-2019,40,All,All,8000,0,10000,9750,10000
7,IBTrACS_p,LogScore,0.87,0.03,0.10,STORM,False,4.749712e+00,STORM+CHAZ+IRIS,1980-2019,40,All,All,8000,0,10000,9750,10000
8,IBTrACS_p,Quantile,1.00,0.00,0.00,STORM,True,4.079038e+07,STORM+CHAZ+IRIS,1980-2019,40,All,All,8000,0,10000,9750,10000
9,IBTrACS_p,"Quantile (0.05, 0.1, 0.2)",0.00,1.00,0.00,CHAZ,True,1.061665e+07,STORM+CHAZ+IRIS,1980-2019,40,All,All,8000,0,10000,9750,10000


## 2. Quantile windows

In [ ]:
results['quantile_windows'] = sweep(data, expand_specs(BASE, target=TARGETS), WINDOWS, min_obs=MIN_OBS)
results['quantile_windows']


3/36 (19s)
6/36 (39s)
9/36 (57s)
12/36 (93s)
15/36 (129s)
18/36 (165s)
21/36 (200s)
24/36 (236s)
27/36 (272s)
30/36 (307s)
33/36 (344s)
36/36 (380s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,years,n_years,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,"Quantile (0.05, 0.1, 0.2)",0.00,1.00,0.00,CHAZ,True,6.116608e+06,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
1,IBTrACS,"Quantile (0.1, 0.2, 0.3)",0.00,1.00,0.00,CHAZ,True,1.047546e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
2,IBTrACS,"Quantile (0.2, 0.3, 0.4)",0.00,1.00,0.00,CHAZ,True,1.565787e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
3,IBTrACS,"Quantile (0.3, 0.4, 0.5)",0.00,1.00,0.00,CHAZ,True,2.065726e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
4,IBTrACS,"Quantile (0.4, 0.5, 0.6)",0.00,1.00,0.00,CHAZ,True,2.520895e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
5,IBTrACS,"Quantile (0.5, 0.6, 0.7)",0.17,0.83,0.00,CHAZ,False,2.898123e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
6,IBTrACS,"Quantile (0.6, 0.7, 0.8)",1.00,0.00,0.00,STORM,True,3.158978e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
7,IBTrACS,"Quantile (0.7, 0.8, 0.9)",1.00,0.00,0.00,STORM,True,3.138129e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
8,IBTrACS,"Quantile (0.8, 0.9, 0.95)",1.00,0.00,0.00,STORM,True,2.852634e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,41,0,10000,9750,10000
9,IBTrACS_p,"Quantile (0.05, 0.1, 0.2)",0.00,1.00,0.00,CHAZ,True,1.061665e+07,STORM+CHAZ+IRIS,1980-2019,40,All,All,8000,0,10000,9750,10000


## 3. Conditional on $X < u$, $u$ = first LogScore bin edge above the largest IBTrACS annual loss
Target years above $u$ are dropped (`n_above_dropped`) and the blend is scored conditional on $X < u$; the weights
reported are those of the full blend, and they carry no information about $P(X > u)$. $u$ is the largest IBTrACS
annual loss over `BASE_YEARS` rounded **up to the next bin edge** (a multiple of `LOG_BIN_WIDTH`), so no LogScore bin
is cut part-way; cutting inside a bin distorts the bin probability of the largest observations.

In [ ]:
U_MAX = max_loss_threshold(data, 'IBTrACS', years=BASE_YEARS)                  # largest IBTrACS annual loss
U = float(np.nextafter((np.floor(U_MAX / LOG_BIN_WIDTH) + 1) * LOG_BIN_WIDTH, -np.inf))  # just below the next bin edge
RULES_3 = {**STANDARD, **WINDOWS}
print(f"largest IBTrACS annual loss = {U_MAX:.4g}, u = {U:.4g}")

results['conditional'] = sweep(data, expand_specs(replace(BASE, condition_below=U), target=TARGETS), RULES_3,
                               min_obs=MIN_OBS)
results['conditional']


u = 6.012e+08
5/52 (27s)
10/52 (59s)
15/52 (103s)
20/52 (452s)
25/52 (514s)
30/52 (601s)
35/52 (943s)
40/52 (1011s)
45/52 (1387s)
50/52 (1450s)
52/52 (1474s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,years,n_years,enso_phase,loss_range,condition_below,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,CRPS,0.51,0.00,0.49,STORM,False,4.138448e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000
1,IBTrACS,LogScore,1.00,0.00,0.00,STORM,True,2.273791e+00,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000
2,IBTrACS,Quantile,0.43,0.00,0.57,IRIS,False,2.159173e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000
3,IBTrACS,"Quantile (0.05, 0.1, 0.2)",0.00,1.00,0.00,CHAZ,True,6.116959e+06,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",0.71,0.29,0.00,STORM,False,2.687622e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000
5,IBTrACS,QuantileWeightedCRPS,0.55,0.00,0.45,STORM,False,1.743045e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000
6,IBTrACS,"Quantile (0.1, 0.2, 0.3)",0.00,1.00,0.00,CHAZ,True,1.047977e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000
7,IBTrACS,"Quantile (0.2, 0.3, 0.4)",0.00,1.00,0.00,CHAZ,True,1.567825e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000
8,IBTrACS,"Quantile (0.3, 0.4, 0.5)",0.00,1.00,0.00,CHAZ,True,2.074715e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000
9,IBTrACS,"Quantile (0.4, 0.5, 0.6)",0.00,0.59,0.41,CHAZ,False,2.545214e+07,STORM+CHAZ+IRIS,1980-2020,41,All,All,6.012146e+08,41,0,10000,9750,10000


## 4. Subsampling
Re-fits on `N_ITER` random subsets of `FRAC` of the calendar years (all counterfactual seasons of a year stay
together), unconditionally and for $X \le u$ (section 3). Columns give the mean, sd and 2.5 / 50 / 97.5% of each
weight, and the share of subsets with a corner solution.

In [ ]:
FRAC, N_ITER = 0.6, 50
RULES_4 = STANDARD

results['subsample'] = subsample_sweep(data, expand_specs(BASE, target=TARGETS, condition_below=[None, U]),
                                       RULES_4, frac=FRAC, n_iter=N_ITER, by='year', min_obs=MIN_OBS)
results['subsample']


1/48 (402s)
2/48 (416s)
3/48 (724s)
4/48 (1031s)
5/48 (1339s)
6/48 (1698s)
7/48 (2098s)
8/48 (2115s)
9/48 (2429s)
10/48 (2716s)
11/48 (3012s)
12/48 (3347s)
13/48 (4204s)
14/48 (4349s)
15/48 (5264s)
16/48 (5741s)
17/48 (6210s)
18/48 (15218s)
19/48 (16382s)
20/48 (16558s)
21/48 (17423s)
22/48 (17858s)
23/48 (18159s)
24/48 (27273s)
25/48 (27840s)
26/48 (27942s)
27/48 (28663s)
28/48 (28975s)
29/48 (29290s)
30/48 (35099s)
31/48 (35643s)
32/48 (35745s)
33/48 (36349s)
34/48 (36653s)
35/48 (36961s)
36/48 (42720s)
37/48 (43295s)
38/48 (43397s)
39/48 (44111s)
40/48 (44415s)
41/48 (44716s)
42/48 (50744s)
43/48 (51297s)
44/48 (51400s)
45/48 (52107s)
46/48 (52410s)
47/48 (52714s)
48/48 (58405s)


,target,scoring_rule,weight_STORM_mean,weight_STORM_sd,weight_STORM_q025,weight_STORM_median,weight_STORM_q975,weight_CHAZ_mean,weight_CHAZ_sd,weight_CHAZ_q025,...,weight_IRIS_median,weight_IRIS_q975,frac_at_corner,components,years,loss_range,condition_below,by,frac,n_iter
0,IBTrACS,CRPS,0.9994,0.004243,1.00000,1.000,1.00000,0.0000,0.000000,0.0,...,0.000,0.00000,0.98,STORM+CHAZ+IRIS,1980-2020,All,NaN,year,0.6,50
1,IBTrACS,LogScore,0.9482,0.108810,0.64125,1.000,1.00000,0.0000,0.000000,0.0,...,0.000,0.35875,0.76,STORM+CHAZ+IRIS,1980-2020,All,NaN,year,0.6,50
2,IBTrACS,Quantile,0.9448,0.114379,0.67000,1.000,1.00000,0.0000,0.000000,0.0,...,0.000,0.33000,0.76,STORM+CHAZ+IRIS,1980-2020,All,NaN,year,0.6,50
3,IBTrACS,"Quantile (0.05, 0.1, 0.2)",0.0000,0.000000,0.00000,0.000,0.00000,1.0000,0.000000,1.0,...,0.000,0.00000,1.00,STORM+CHAZ+IRIS,1980-2020,All,NaN,year,0.6,50
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",0.9456,0.117389,0.69000,1.000,1.00000,0.0000,0.000000,0.0,...,0.000,0.31000,0.82,STORM+CHAZ+IRIS,1980-2020,All,NaN,year,0.6,50
5,IBTrACS,QuantileWeightedCRPS,0.9988,0.003283,0.99000,1.000,1.00000,0.0000,0.000000,0.0,...,0.000,0.01000,0.88,STORM+CHAZ+IRIS,1980-2020,All,NaN,year,0.6,50
6,IBTrACS,CRPS,0.5036,0.433414,0.00000,0.550,1.00000,0.0288,0.106131,0.0,...,0.450,1.00000,0.60,STORM+CHAZ+IRIS,1980-2020,All,6.012146e+08,year,0.6,50
7,IBTrACS,LogScore,0.6476,0.394568,0.00000,0.775,1.00000,0.3524,0.394568,0.0,...,0.000,0.00000,0.60,STORM+CHAZ+IRIS,1980-2020,All,6.012146e+08,year,0.6,50
8,IBTrACS,Quantile,0.5064,0.405778,0.00000,0.445,1.00000,0.0360,0.143413,0.0,...,0.380,1.00000,0.46,STORM+CHAZ+IRIS,1980-2020,All,6.012146e+08,year,0.6,50
9,IBTrACS,"Quantile (0.05, 0.1, 0.2)",0.0000,0.000000,0.00000,0.000,0.00000,1.0000,0.000000,1.0,...,0.000,0.00000,1.00,STORM+CHAZ+IRIS,1980-2020,All,6.012146e+08,year,0.6,50


## 5. Model pairs
Every 2-model blend (STORM+CHAZ, STORM+IRIS, CHAZ+IRIS) refitted for every target: standard scores, quantile
windows, and the section 3 conditional ($X \le u$). Results go to `results['pairs_standard']`,
`results['pairs_quantile_windows']` and `results['pairs_conditional']` and are saved with everything else.
`min_mean_score` is directly comparable with the 3-model run for the same target, rule and condition, so the
difference is what dropping a model costs (the viewer tabulates this).

In [ ]:
PAIRS = model_subsets(COMPONENTS, sizes=(2,))      # [('STORM', 'CHAZ'), ('STORM', 'IRIS'), ('CHAZ', 'IRIS')]
print(PAIRS)

results['pairs_standard'] = sweep(data, expand_specs(BASE, target=TARGETS, components=PAIRS), STANDARD, min_obs=MIN_OBS)
results['pairs_standard']


[('STORM', 'CHAZ'), ('STORM', 'IRIS'), ('CHAZ', 'IRIS')]
7/72 (1s)
14/72 (3s)
21/72 (5s)
28/72 (14s)
35/72 (23s)
42/72 (37s)
49/72 (45s)
56/72 (54s)
63/72 (63s)
70/72 (71s)
72/72 (77s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,years,n_years,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,CRPS,1.0,0.00,NaN,STORM,True,4.221732e+07,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
1,IBTrACS,LogScore,1.0,0.00,NaN,STORM,True,2.271615e+00,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
2,IBTrACS,Quantile,1.0,0.00,NaN,STORM,True,2.178028e+07,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
3,IBTrACS,"Quantile (0.05, 0.1, 0.2)",0.0,1.00,NaN,CHAZ,True,6.116608e+06,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",1.0,0.00,NaN,STORM,True,2.852634e+07,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,MINI-WNC-QM,LogScore,NaN,0.06,0.94,IRIS,False,3.116566e+00,CHAZ+IRIS,1980-2019,40,All,All,8000,0,NaN,9750.0,10000.0
68,MINI-WNC-QM,Quantile,NaN,0.00,1.00,IRIS,True,3.214619e+07,CHAZ+IRIS,1980-2019,40,All,All,8000,0,NaN,9750.0,10000.0
69,MINI-WNC-QM,"Quantile (0.05, 0.1, 0.2)",NaN,1.00,0.00,CHAZ,True,7.850441e+06,CHAZ+IRIS,1980-2019,40,All,All,8000,0,NaN,9750.0,10000.0
70,MINI-WNC-QM,"Quantile (0.8, 0.9, 0.95)",NaN,0.00,1.00,IRIS,True,5.462346e+07,CHAZ+IRIS,1980-2019,40,All,All,8000,0,NaN,9750.0,10000.0


In [ ]:
results['pairs_quantile_windows'] = sweep(data, expand_specs(BASE, target=TARGETS, components=PAIRS), WINDOWS,
                                          min_obs=MIN_OBS)
results['pairs_quantile_windows']


10/108 (2s)
20/108 (4s)
30/108 (7s)
40/108 (10s)
50/108 (13s)
60/108 (16s)
70/108 (18s)
80/108 (20s)
90/108 (22s)
100/108 (25s)
108/108 (27s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,years,n_years,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,"Quantile (0.05, 0.1, 0.2)",0.0,1.0,NaN,CHAZ,True,6.116608e+06,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
1,IBTrACS,"Quantile (0.1, 0.2, 0.3)",0.0,1.0,NaN,CHAZ,True,1.047546e+07,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
2,IBTrACS,"Quantile (0.2, 0.3, 0.4)",0.0,1.0,NaN,CHAZ,True,1.565787e+07,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
3,IBTrACS,"Quantile (0.3, 0.4, 0.5)",0.0,1.0,NaN,CHAZ,True,2.065726e+07,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
4,IBTrACS,"Quantile (0.4, 0.5, 0.6)",0.0,1.0,NaN,CHAZ,True,2.520895e+07,STORM+CHAZ,1980-2020,41,All,All,41,0,10000.0,9750.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,MINI-WNC-QM,"Quantile (0.4, 0.5, 0.6)",NaN,1.0,0.0,CHAZ,True,3.307549e+07,CHAZ+IRIS,1980-2019,40,All,All,8000,0,NaN,9750.0,10000.0
104,MINI-WNC-QM,"Quantile (0.5, 0.6, 0.7)",NaN,1.0,0.0,CHAZ,True,3.878977e+07,CHAZ+IRIS,1980-2019,40,All,All,8000,0,NaN,9750.0,10000.0
105,MINI-WNC-QM,"Quantile (0.6, 0.7, 0.8)",NaN,1.0,0.0,CHAZ,True,4.483677e+07,CHAZ+IRIS,1980-2019,40,All,All,8000,0,NaN,9750.0,10000.0
106,MINI-WNC-QM,"Quantile (0.7, 0.8, 0.9)",NaN,0.0,1.0,IRIS,True,5.015321e+07,CHAZ+IRIS,1980-2019,40,All,All,8000,0,NaN,9750.0,10000.0


In [ ]:
results['pairs_conditional'] = sweep(data, expand_specs(replace(BASE, condition_below=U), target=TARGETS, components=PAIRS),
                                     RULES_3, min_obs=MIN_OBS)
results['pairs_conditional']


15/156 (2s)
30/156 (4s)
45/156 (10s)
60/156 (18s)
75/156 (25s)
90/156 (32s)
105/156 (39s)
120/156 (47s)
135/156 (54s)
150/156 (65s)
156/156 (66s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,years,n_years,enso_phase,loss_range,condition_below,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,CRPS,0.31,0.69,NaN,CHAZ,False,4.140621e+07,STORM+CHAZ,1980-2020,41,All,All,6.012146e+08,41,0,10000.0,9750.0,NaN
1,IBTrACS,LogScore,1.00,0.00,NaN,STORM,True,2.273791e+00,STORM+CHAZ,1980-2020,41,All,All,6.012146e+08,41,0,10000.0,9750.0,NaN
2,IBTrACS,Quantile,0.45,0.55,NaN,CHAZ,False,2.160428e+07,STORM+CHAZ,1980-2020,41,All,All,6.012146e+08,41,0,10000.0,9750.0,NaN
3,IBTrACS,"Quantile (0.05, 0.1, 0.2)",0.00,1.00,NaN,CHAZ,True,6.116959e+06,STORM+CHAZ,1980-2020,41,All,All,6.012146e+08,41,0,10000.0,9750.0,NaN
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",0.71,0.29,NaN,STORM,False,2.687622e+07,STORM+CHAZ,1980-2020,41,All,All,6.012146e+08,41,0,10000.0,9750.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,MINI-WNC-QM,"Quantile (0.3, 0.4, 0.5)",NaN,1.00,0.00,CHAZ,True,1.692602e+07,CHAZ+IRIS,1980-2019,40,All,All,6.012146e+08,7834,166,NaN,9750.0,10000.0
152,MINI-WNC-QM,"Quantile (0.4, 0.5, 0.6)",NaN,0.97,0.03,CHAZ,False,2.086008e+07,CHAZ+IRIS,1980-2019,40,All,All,6.012146e+08,7834,166,NaN,9750.0,10000.0
153,MINI-WNC-QM,"Quantile (0.5, 0.6, 0.7)",NaN,0.00,1.00,IRIS,True,2.418635e+07,CHAZ+IRIS,1980-2019,40,All,All,6.012146e+08,7834,166,NaN,9750.0,10000.0
154,MINI-WNC-QM,"Quantile (0.6, 0.7, 0.8)",NaN,0.63,0.37,CHAZ,False,2.614818e+07,CHAZ+IRIS,1980-2019,40,All,All,6.012146e+08,7834,166,NaN,9750.0,10000.0


In [ ]:
# Optional: subsampling for the pairs (same settings as section 4); uncomment to run.
# results['pairs_subsample'] = subsample_sweep(data, expand_specs(BASE, target=TARGETS, components=PAIRS), RULES_4,
#                                              frac=FRAC, n_iter=N_ITER, by='year', min_obs=MIN_OBS)


## 6. Save all results

In [12]:
for tag, df in results.items():
    save_run(df, tag, out_dir=RUNS_DIR, notes=NOTES)
pd.read_csv(f'{RUNS_DIR}/manifest.csv').tail(len(results))


saved 72 rows -> results\blend_runs\20261030_014853__pairs_standard
saved 108 rows -> results\blend_runs\20261030_014853__pairs_quantile_windows
saved 156 rows -> results\blend_runs\20261030_014853__pairs_conditional


,timestamp,tag,n_rows,folder,notes
9,20261030_014853,pairs_standard,72,results\blend_runs\20261030_014853__pairs_stan...,NaN
10,20261030_014853,pairs_quantile_windows,108,results\blend_runs\20261030_014853__pairs_quan...,NaN
11,20261030_014853,pairs_conditional,156,results\blend_runs\20261030_014853__pairs_cond...,NaN
